# Prática 01 — Caminho Mínimo (Dijkstra vs Heurística Gulosa)

> Objetivo: comparar o algoritmo de Dijkstra com uma solução gulosa para SSSP (origem única, múltiplos destinos) em grafos direcionados ponderados com pesos positivos.

## Escopo da avaliação
1) Parte 1 — Dijkstra
- Implementação de Dijkstra com contagem de comparações de relaxamento.
- Testes:
  - (a) Grafos completos com 4,5,6,...,N vértices (pesos > 0).
  - (b) Estrutura de dados: listas de adjacência.
  - (c) Executar com origem 0 (ou outro vértice qualquer).
  - (d) Contar comparações.
  - (e) Plotar gráfico (X = n, Y = comparações).
  - (f) Aplicar em instâncias do Moodle com 10.000 e 1.000.000 vértices; origem = 0; exibir comparações.

2) Parte 2 — Heurística Gulosa
- Implementação de uma heurística gulosa baseada em escolher sempre a aresta mais leve que sai do conjunto alcançado.
- Mesmos testes e entregáveis da Parte 1.

> Observações:
- N máximo teórico: 1.000.000 (para grafos completos, isso é impraticável em memória; use N pequenos para o benchmark).
- Para as instâncias do Moodle, execute apenas a célula específica de instância.

## Como usar
- Execute as células na ordem: importações → utilitários → Dijkstra → Gulosa → funções auxiliares → execução em instância → benchmark opcional.
- Ajuste caminhos das instâncias se necessário (por padrão, espera arquivos nesta pasta).

In [ ]:
# Importações principais
import os
import random
import matplotlib.pyplot as plt

# Todas as implementações estão neste notebook (sem dependências externas)
# %matplotlib inline  # Em alguns ambientes é implícito; descomente se necessário.

In [ ]:
# Utilitários de entrada/geração de grafos (autocontidos)
from typing import Dict, List, Tuple, Optional, Callable
import random as _random
Adj = Dict[int, List[Tuple[int, float]]]

def read_graph_file(path: str, directed: bool = True) -> Tuple[int, Adj]:
    """
    Lê um grafo do arquivo no formato:
      1a linha: n (número de vértices)
      2a linha: m (número de arestas)
      próximas m linhas: u v w (índices e peso real)
    Retorna (n, adj), onde adj[u] = lista de (v, w).
    Se directed=False, adiciona a aresta reversa.
    """
    adj: Adj = {}
    with open(path, 'r', encoding='utf-8') as f:
        header = f.readline()
        if not header:
            raise ValueError('Arquivo vazio')
        n = int(header.strip())
        for u in range(n):
            adj[u] = []
        m_line = f.readline()
        if not m_line:
            raise ValueError('Arquivo sem linha de arestas (m)')
        m = int(m_line.strip())
        for _ in range(m):
            line = f.readline()
            if not line:
                break
            parts = line.strip().split()
            if len(parts) < 3:
                continue
            u, v = int(parts[0]), int(parts[1])
            w = float(parts[2])
            adj[u].append((v, w))
            if not directed:
                adj[v].append((u, w))
    return n, adj

def make_complete_graph(n: int,
                        weight_fn: Optional[Callable[[int, int], float]] = None,
                        directed: bool = True) -> Adj:
    """
    Cria um grafo completo com pesos positivos. CUIDADO: usa O(n^2) memória.
    - Se weight_fn=None, usa peso positivo pseudo-aleatório determinístico.
    - Se directed=True, cria arestas (u->v) para u!=v.
    - Se directed=False, cria u<v e adiciona nas duas direções.
    """
    if weight_fn is None:
        def weight_fn(u: int, v: int) -> float:
            return (_random.random() + 0.0001)  # > 0
    adj: Adj = {i: [] for i in range(n)}
    if directed:
        for u in range(n):
            for v in range(n):
                if u == v:
                    continue
                w = float(abs(weight_fn(u, v)))
                if w == 0.0:
                    w = 1e-9
                adj[u].append((v, w))
    else:
        for u in range(n):
            for v in range(u+1, n):
                w = float(abs(weight_fn(u, v)))
                if w == 0.0:
                    w = 1e-9
                adj[u].append((v, w))
                adj[v].append((u, w))
    return adj

In [ ]:
# Dijkstra com contagem de comparações (autocontido)
from math import inf
import heapq
from typing import Dict, List, Tuple, Optional

class DijkstraResult:
    def __init__(self, dist: Dict[int, float], parent: Dict[int, Optional[int]], comparisons: int):
        self.dist = dist
        self.parent = parent
        self.comparisons = comparisons  # comparações de relaxamento (dist[u] + w < dist[v])

def dijkstra_count(adj: Dict[int, List[Tuple[int, float]]], s: int) -> DijkstraResult:
    """
    Algoritmo de Dijkstra com contagem de comparações de relaxamento.
    - Pesos devem ser >= 0.
    - Retorna dist, parent e total de comparações (if dist[u] + w < dist[v]).
    """
    dist: Dict[int, float] = {u: inf for u in adj}
    parent: Dict[int, Optional[int]] = {u: None for u in adj}
    dist[s] = 0.0

    # checagem leve de pesos negativos
    for u, nbrs in adj.items():
        for v, w in nbrs:
            if w < 0:
                raise ValueError('Dijkstra requer pesos não negativos')

    heap: List[Tuple[float, int]] = [(0.0, s)]
    visited = set()
    comparisons = 0

    while heap:
        du, u = heapq.heappop(heap)
        if u in visited:
            continue
        visited.add(u)
        for v, w in adj[u]:
            comparisons += 1
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                parent[v] = u
                heapq.heappush(heap, (dist[v], v))

    return DijkstraResult(dist, parent, comparisons)

In [ ]:
# Heurística gulosa (autocontida)
from typing import Dict, List, Tuple, Optional
from math import inf
import heapq

class GreedyResult:
    def __init__(self, dist: Dict[int, float], parent: Dict[int, Optional[int]], comparisons: int):
        self.dist = dist
        self.parent = parent
        self.comparisons = comparisons

def greedy_sssp_edge_first(adj: Dict[int, List[Tuple[int, float]]], s: int) -> GreedyResult:
    """
    Heurística gulosa para SSSP: mantém um conjunto de vértices alcançados
    e uma heap de arestas que saem de alcançados; escolhe sempre a aresta de menor peso.
    Contagem: apenas comparações de relaxamento (dist[u] + w < dist[v]).
    Obs.: Não é correta em geral para SSSP; é usada para comparar contagens.
    """
    dist: Dict[int, float] = {u: inf for u in adj}
    parent: Dict[int, Optional[int]] = {u: None for u in adj}
    dist[s] = 0.0

    visited = set([s])
    edge_heap: List[Tuple[float, int, int]] = []  # (w, u, v)
    for v, w in adj[s]:
        heapq.heappush(edge_heap, (w, s, v))

    comparisons = 0
    while edge_heap:
        w, u, v = heapq.heappop(edge_heap)
        comparisons += 1
        if dist[u] + w < dist[v]:
            dist[v] = dist[u] + w
            parent[v] = u
            if v not in visited:
                visited.add(v)
                for x, wx in adj[v]:
                    heapq.heappush(edge_heap, (wx, v, x))

    return GreedyResult(dist, parent, comparisons)

## Funções auxiliares
Estas funções reproduzem a lógica de benchmark/plotagem e a aplicação em instâncias de arquivo.

In [ ]:
from typing import List, Tuple

def bench_complete_graphs(Nmax: int, step: int = 1, seed: int = 42, directed: bool = True):
    """
    Gera grafos completos de 4 até Nmax (passo 'step') e contabiliza as comparações
    de relaxamento em Dijkstra e na heurística gulosa.
    CUIDADO: grafo completo é O(n^2) em memória.
    """
    random.seed(seed)
    ns: List[int] = []
    comps_dij: List[int] = []
    comps_greedy: List[int] = []
    for n in range(4, Nmax + 1, step):
        adj = make_complete_graph(n, directed=directed)
        s = 0
        res_d = dijkstra_count(adj, s)
        res_g = greedy_sssp_edge_first(adj, s)
        ns.append(n)
        comps_dij.append(res_d.comparisons)
        comps_greedy.append(res_g.comparisons)
        print(f"n={n} | comps: dijkstra={res_d.comparisons} greedy={res_g.comparisons}")
    return ns, comps_dij, comps_greedy

def plot_comparisons(ns: List[int], comps_dij: List[int], comps_greedy: List[int]):
    plt.figure(figsize=(10,6))
    plt.plot(ns, comps_dij, label='Dijkstra (comparações)', linewidth=2)
    plt.plot(ns, comps_greedy, label='Greedy (comparações)', linewidth=2)
    plt.xlabel('Número de vértices (n)')
    plt.ylabel('Número de comparações (relax)')
    plt.title('Comparações vs n (grafos completos)')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.4)
    plt.show()

def count_on_instance(path: str, origin: int = 0):
    n, adj = read_graph_file(path, directed=True)
    print(f"Lida instância {path} com n={n}")
    res_d = dijkstra_count(adj, origin)
    res_g = greedy_sssp_edge_first(adj, origin)
    print(f"Comparações (Dijkstra) = {res_d.comparisons}")
    print(f"Comparações (Greedy)   = {res_g.comparisons}")
    return res_d, res_g

## Execução em instância de arquivo
Por padrão, este exemplo tenta usar `10000.txt` localizado nesta mesma pasta.
Altere `origin` conforme necessário.

In [ ]:
# Caminho padrão para a instância local (10.000 vértices)
instance_path = '10000.txt'  # ajuste se estiver executando de outra pasta
origin = 0

if os.path.exists(instance_path):
    res_d, res_g = count_on_instance(instance_path, origin=origin)
else:
    print('Arquivo não encontrado:', instance_path)
    print('Verifique o caminho corrente do notebook ou ajuste a variável instance_path.')

## Instância com 1.000.000 de vértices (opcional e pesado)
Ative a execução definindo `RUN_LARGE = True`. Certifique-se de ter memória suficiente. O arquivo esperado é:
`largeEWD - contains one million vertices and 15172126 edges.txt`.

In [ ]:
# Execução controlada da instância grande (desativada por padrão)
RUN_LARGE = False
large_path = 'largeEWD - contains one million vertices and 15172126 edges.txt'
origin = 0

if RUN_LARGE:
    if os.path.exists(large_path):
        _ = count_on_instance(large_path, origin=origin)
    else:
        print('Arquivo não encontrado:', large_path)
else:
    print('Execução desativada: defina RUN_LARGE = True para rodar a instância grande.')

## Benchmark em grafos completos pequenos (opcional)
Cuidado: grafos completos consomem O(n^2) de memória; mantenha N pequeno.

In [ ]:
# Parâmetros do benchmark (mantenha pequenos para evitar uso excessivo de memória)
Nmax = 50
step = 5

ns, c_d, c_g = bench_complete_graphs(Nmax, step=step, directed=True)
plot_comparisons(ns, c_d, c_g)

### Observações
- As instâncias fornecidas costumam listar arestas para ambos os sentidos; usamos `directed=True` na leitura para não duplicar.
- Para rodar na instância com ~1M de vértices, execute apenas a célula de instância e evite o benchmark de grafo completo.
- A heurística gulosa não é correta no geral para SSSP; é usada aqui apenas para comparação de contagens.